<a href="https://colab.research.google.com/github/savneetbadial/Wide-Receiver-Stats-Model/blob/tpu/TPU_Wide_Receiver_Stats_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Define your file path in Google Drive
# Adjust the folder path if you put it somewhere else
file_path = '/content/drive/My Drive/Colab Notebooks/NFL/wr_games_raw.csv'

# Load the dataset directly
df = pd.read_csv(file_path)
print(f"Successfully loaded {len(df):,} rows from Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully loaded 27,962 rows from Google Drive!


In [3]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 16.8 MB/s  0:00:19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 66.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [tensorflow]


In [5]:
import tensorflow as tf

# Initialize TPU for single-core v5e-1 TPU runtime in Colab
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    print("Successfully connected to TPU v5e-1:", resolver.master())
except Exception as e:
    print(f"TPU initialization note: {e}")
    # Fallback for single-core Colab TPU runtimes
    try:
        import jax
        print("JAX TPU devices:", jax.devices())
    except:
        pass
    strategy = tf.distribute.get_strategy()

print(f"Replicas in sync: {strategy.num_replicas_in_sync}")

TPU initialization note: Please provide a TPU Name to connect to.
JAX TPU devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Replicas in sync: 1


In [13]:
# ==========================================
# 1. ENVIRONMENT & BACKEND SETUP
# ==========================================
import os
os.environ["KERAS_BACKEND"] = "jax"

import time
import psutil
import keras
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Initializing JAX backend for TPU v5e-1...")
try:
    import jax
    print("-> JAX TPU devices detected:", jax.devices())
except Exception as e:
    print(f"-> JAX device check note: {e}")

# ==========================================
# 2. DATA PREPARATION & BATCHING
# ==========================================
features = [
    'home', 'targets_avg3', 'rec_avg3', 'yds_avg3', 'td_avg3',
    'targets_avg8', 'rec_avg8', 'yds_avg8', 'yac_avg8',
    'team_pass_att_avg3', 'opp_wr_yds_allowed_avg5', 'opp_pass_def_rank',
    'games_played_career'
]
target = 'receiving_yards'

X = df[features].fillna(0).values
y = df[target].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

BATCH_SIZE = 256

train_dataset = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ==========================================
# 3. MODEL ARCHITECTURE & COMPILATION
# ==========================================
model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],), name='input_features'),
    keras.layers.Dense(128, activation='relu', name='hidden_layer_1'),
    keras.layers.Dense(64, activation='relu', name='hidden_layer_2'),
    keras.layers.Dense(32, activation='relu', name='hidden_layer_3'),
    keras.layers.Dense(1, name='output_receiving_yards')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

model.summary()

# ==========================================
# 4. TPU TRAINING & BENCHMARKING
# ==========================================
print("\nStarting TPU v5e-1 Training via JAX Backend...")
start_time = time.time()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30
)

tpu_training_time = time.time() - start_time

# ==========================================
# 5. COMPREHENSIVE PERFORMANCE METRICS
# ==========================================
print("\n" + "="*60)
print("--- COMPREHENSIVE HARDWARE & MODEL PERFORMANCE METRICS ---")
print("="*60)

# Training Throughput & Efficiency
total_samples = len(X_train)
samples_per_sec = total_samples / (tpu_training_time / 30)
print(f"• Training Throughput: {samples_per_sec:.2f} samples/sec")
print(f"• Total Training Time (30 epochs): {tpu_training_time:.2f} seconds")
print(f"• Average Epoch Time: {tpu_training_time / 30:.4f} seconds/epoch")

# Inference Latency
sample_batch = X_val[:32]
_ = model.predict(sample_batch, verbose=0)  # Warmup run

start_inf = time.time()
num_iterations = 100
for _ in range(num_iterations):
    _ = model.predict(sample_batch, verbose=0)
end_inf = time.time()

avg_batch_latency_ms = ((end_inf - start_inf) / num_iterations) * 1000
single_game_latency_ms = avg_batch_latency_ms / 32

print(f"• Inference Latency (Batch of 32 games): {avg_batch_latency_ms:.4f} ms")
print(f"• Inference Latency (Single game prediction): {single_game_latency_ms:.4f} ms")

# Model Resource Footprint
trainable_count = sum([np.prod(w.shape) for w in model.trainable_weights])
print(f"• Total Trainable Parameters: {trainable_count:,}")
print(f"• Approximate Model Memory Footprint: {(trainable_count * 4) / 1024:.2f} KB (weights only)")

# Regression Accuracy Metrics
val_loss, val_mae = model.evaluate(val_dataset, verbose=0)
print(f"• Validation Loss (MSE): {val_loss:.2f}")
print(f"• Validation MAE: {val_mae:.2f} receiving yards")

# Classification Metrics (Breakout Game Classification: >= 70 receiving yards)
y_val_preds = model.predict(X_val, verbose=0).flatten()
breakout_threshold = 70.0
y_true_binary = (y_val >= breakout_threshold).astype(int)
y_pred_binary = (y_val_preds >= breakout_threshold).astype(int)

acc = accuracy_score(y_true_binary, y_pred_binary)
prec = precision_score(y_true_binary, y_pred_binary, zero_division=0)
rec = recall_score(y_true_binary, y_pred_binary, zero_division=0)
f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)

print(f"\n--- BREAKOUT GAME CLASSIFICATION METRICS (>= 70 Yards) ---")
print(f"• Classification Accuracy: {acc * 100:.2f}%")
print(f"• Precision: {prec:.4f}")
print(f"• Recall: {rec:.4f}")
print(f"• F1-Score: {f1:.4f}")
print("="*60)

Initializing JAX backend for TPU v5e-1...
-> JAX TPU devices detected: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer_1 (Dense)          │ (None, 128)            │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_3 (Dense)          │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_receiving_yards (Dense)  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,161 (47.50 KB)

 Trainable params: 12,161 (47.50 KB)

 Non-trainable params: 0 (0.00 B)


Starting TPU v5e-1 Training via JAX Backend...
Epoch 1/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1971.3114 - mae: 31.5991 - val_loss: 1176.7200 - val_mae: 25.4763
Epoch 2/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1140.3301 - mae: 25.4224 - val_loss: 1117.4960 - val_mae: 25.1113
Epoch 3/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1094.7784 - mae: 24.9034 - val_loss: 1095.4585 - val_mae: 24.9164
Epoch 4/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1075.6027 - mae: 24.5766 - val_loss: 1083.7994 - val_mae: 24.6946
Epoch 5/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1067.0980 - mae: 24.4521 - val_loss: 1077.8199 - val_mae: 24.7022
Epoch 6/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1060.8267 - mae: 24.3983 - val_loss: 1074.1130 - val_mae: 24.8410
Epoch 7/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1057.9792 - mae: 24.3898 - val_loss: 1069.9979 - val_mae: 24.4062
Epoch 8/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1055.3640 - mae: 24.3297 -